<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/07_value_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Value Function

In this notebook, we're going to explore the value function and a few simple design options

In [ ]:
from functools import partial
from typing import Callable

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation
from matplotlib.patches import Rectangle

# Set up a "key" for any random operation
key = jax.random.PRNGKey(0)

In this problem, we will be spectating two drivers performing a series of "races" and their performance will be determined by three **judges**. Given our understanding of value functions, and cost functions, we, as a spectator, would like to predict who will be the winner of the races given we have some knowledge about what each judge prefers.

Each driver has their unique driving style. That is, they have their own unique control policy.

Our "car" is the double integrator, and its goal is to stop perfectly at the finish line. We'll use the Value Function (also called "cost-to-go") as our scorecard to see which driver is more "skilled" at this task.

## (a) System
First, let's define the "car" and the "track" it drives on. We'll use a discrete-time version of the car's dynamics with timestep size $ \Delta t $. Given a state $ x_k $ and control $ u_k $ at step $ k $, the next state $ x_{k+1} $ is:

\begin{aligned}
p_{k+1} = p_k + v_k \cdot \Delta t\\
v_{k+1} = v_k + u_k \cdot \Delta t,
\end{aligned}
where $ p $ is the position and $ v $ is car's velocity.

The driver controls the car using $ u $, which is the accelaration (the gas/brake pedal). The drivers's goal is to get the car to the state $ x =0 $, which means stopped ($ v = 0 $) at the finish line ($ p =0 $).

In [ ]:
@jax.jit
def car_dynamics(state: jnp.array, control: float, dt: float):
    """
    Computes the next state of the double integrator.

    Args:
      state: jnp.array, current state [position, velocity]
      control: float, current control (acceleration)
      dt: float, time step

    Returns:
      jnp.array, next state [position_next, velocity_next]
    """
    # exctract the states
    position, velocity = state

    # compute the next state
    position_next = position + velocity * dt
    velocity_next = velocity + control * dt

    return jnp.array([position_next, velocity_next])

## (b) The "Scorecard" (Value Function)
As a judge for this problem, how do we know if a driver (policy) is "good"?

We need a scorecard!
**Penalty (stage cost)**: At each time step, the driver gets 'penalty points' based on their position, velocity and control effort. This is the *stage cost*:
\begin{aligned}
L(x,u) = x^\top Q x + Ru^2 = q_p . p^2 + q_v.v^2 + r.u^2
\end{aligned}
where $ x^\top Q x $ is the error penalty as the driver get penalized for being far from the finish line or for going too fast; $ Ru^2 $ is the effort penalty as the driver get penalized for using the gas/brake too much.

As the judge, you get to decide the relative importance between getting close to the finish line, the speed, and the control effort. You can express your own judging preferences through the values of $ Q $ and $ R $.

**Final score (value function)**: The value function, $V_{\pi}(x_0)$, is the total penalty score for an entire drive, starting from $x_0$ following policy $\pi$. A lower score is better!

\begin{aligned}
V_{\pi}(x_0) = \sum_{k=0}^{N-1} L(x_k, \pi(x_k))\cdot \Delta t.
\end{aligned}
Let's set up some functions in order to build our "scorecard".

In [ ]:
# TODO by student

@jax.jit
def stage_cost(state: jnp.array, control: float, Q: jnp.array, R: float):
    """
    Calculates the stage cost L(state, control) = state^T Q state + R control^2.
    """
    # TODO by student
    # calculate the state cost
    # calculate the control cost
    # return the sum of state cost and control cost
    return NotImplementedError("stage_cost function not implemented yet")


@partial(jax.jit, static_argnames=("policy_fn", "N"))
def compute_value_and_rollout(
    policy_fn: Callable[[jnp.array], jnp.array],
    initial_state: jnp.array,
    Q: jnp.array,
    R: float,
    dt: float,
    N: int,
):
    """
    Simulates the system and computes the total cost-to-go.

    Args:
      policy_fn: A function (policy) that takes state x and returns control u.
      initial_state: Initial state [p, v].
      Q: State cost matrix (2x2).
      R: Control cost scalar.
      dt: Time step.
      N: Number of steps.

    Returns:
      total_cost: The V(initial_state) value.
      history: A tuple (x_hist, u_hist)
    """

    # Below is basically a for-loop but implemented with jax.lax.scan for efficiency.
    # This inner function is what jax.lax.scan will call at each step.
    def simulation_step(current_x, _):

        # TODO by student
        # 1. Get control from policy
        current_u = ...

        # 2. Calculate the cost for this step
        cost = ...

        # 3. Get the next state from dynamics
        next_x = ...

        # 4. Return the next state
        return NotImplementedError("simulation_step function not implemented yet")

    # Run the simulation loop
    # `scan` takes the step_function, the initial state (x0),
    # and an array of "inputs" (we just use zeros as placeholders)
    final_state, history_data = jax.lax.scan(f=simulation_step, init=initial_state, xs=jnp.zeros(N))

    x_hist, u_hist, cost_hist = history_data

    # Sum up all the costs and multiply by dt
    total_cost = jnp.sum(cost_hist) * dt

    return total_cost, (x_hist, u_hist)

## (c) Meet the "Drivers" (Policies)

We'll introduce our two drivers.

To make it a bit more realistic, our car has a control limit. The driver can't accelerate or brake infinitely hard. We'll limit the control to $ u_{\max}=3.0 $.

In [ ]:
@jax.jit
def saturate(u, u_max):
    """Clips the control input to [-u_max, u_max]."""
    return jnp.clip(u, -u_max, u_max)


# TODO by student: Implement the policy
@jax.jit
def policy_general(state, position_gain, velocity_gain, u_max):
    """
    The aggressive policy.
    Policy: u = - pos_gain * p - vel_gain * v
    """
    # Extract position and velocity from state
    # Calculate control input using position and velocity gains
    # Apply saturation to the control input

    raise NotImplementedError("policy_general function not implemented yet")

Consider two types of drivers

- Driver 1: This driver is aggressive. They always want to correct errors fast.

- Driver 2: This driver is cautious and gentle on the pedals.

In [ ]:
u_max = 3.0  # Max acceleration/braking

# driver 1 gains and policy
position_gain_driver1 = 3.0
velocity_gain_driver1 = 2.0
policy_driver1 = ...

# driver 2 gains and policy
position_gain_driver2 = 0.5
velocity_gain_driver2 = 1.0
policy_driver2 = ...

## (d) Warm-up

Before the actual race, we watch each driver warm up, and we can observe how well they do at various starting points. While the judges aren't ready yet to give their scores, you can make up your own judging criteria and see how the drivers perform.

As the thorough and rigourous engineer that you are, you pay attention the driver's warm up routines, and consider who would be considered the winner given different possible judging criteria.


 [TODO by student]

Test out different starting states and different judging preferences. Describe the trends/behaviors that you observe.

[student response here]

In [ ]:
# starting state
start_state = jnp.array([-5.0, 0.0]) # change me


# cost parameters
# pick something for now, just to see results
Q = jnp.diag(jnp.array([10.0, 1.0])) # change me
R = 0.0 # change me

# Simulation parameters
dt = 0.05  # Time step
T = 10.0  # Total simulation time
n_steps = int(T / dt)  # Number of steps


V_test_1, (x_hist_1, u_hist_1) = compute_value_and_rollout(policy_driver1, start_state, Q, R, dt, n_steps)
print(f"Driver 1 (Aggressive) Test Value: {V_test_1:.2f}")

V_test_2, (x_hist_2, u_hist_2) = compute_value_and_rollout(policy_driver2, start_state, Q, R, dt, n_steps)
print(f"Driver 2 (Gentle) Test Value: {V_test_2:.2f}")

In [ ]:
# animate race
def plot_car(
    ax,
    position,
    lane,
    car_color="blue",
    car_width=0.5,
    car_height=0.3,
    label="Driver 1",
):
    # plot the car as a rectangle
    car = Rectangle(
        (position - car_width / 2, lane - car_height / 2),
        car_width,
        car_height,
        color=car_color,
        label=label,
    )
    ax.add_patch(car)


def plot_acceleration(ax, position, lane, acceleration, scale=0.5, color="red"):
    # plot the acceleration as an arrow
    ax.arrow(
        position,
        lane,
        acceleration * scale,
        0,
        head_width=0.1,
        head_length=0.1,
        fc=color,
        ec=color,
    )


# plot the lanes
top_lane = 0.5
bottom_lane = -0.5
fig, ax = plt.subplots(figsize=(14, 3))


def update(time_index):
    ax.clear()
    plot_car(ax, x_hist_1[time_index, 0], lane=bottom_lane, car_color="C0", label="Driver 1")
    plot_car(ax, x_hist_2[time_index, 0], lane=top_lane, car_color="C1", label="Driver 2")
    plot_acceleration(
        ax,
        x_hist_1[time_index, 0],
        lane=bottom_lane,
        acceleration=u_hist_1[time_index],
        color="C0",
    )
    plot_acceleration(
        ax,
        x_hist_2[time_index, 0],
        lane=top_lane,
        acceleration=u_hist_2[time_index],
        color="C1",
    )
    ax.set_xlim(start_state[0] - 2, max(x_hist_1[:, 0].max(), x_hist_2[:, 0].max()) + 2)
    ax.set_ylim(-2, 2)
    ax.set_title(f"Time: {time_index * dt:.2f}s")
    ax.vlines(
        0,
        -5,
        5,
        colors="red",
        linestyles="--",
        label="Goal Position",
        zorder=0,
        alpha=0.5,
    )
    ax.vlines(
        start_state[0],
        -5,
        5,
        colors="forestgreen",
        linestyles="--",
        label="Start Position",
        zorder=0,
        alpha=0.5,
    )
    ax.hlines(
        bottom_lane,
        start_state[0] - 2,
        5,
        colors="C0",
        linestyles="--",
        zorder=0,
        alpha=0.5,
    )
    ax.hlines(
        top_lane,
        start_state[0] - 2,
        5,
        colors="C1",
        linestyles="--",
        zorder=0,
        alpha=0.5,
    )
    ax.grid(alpha=0.3)
    ax.legend()
    return []

In [ ]:
# animate racw
ani = animation.FuncAnimation(fig, update, frames=x_hist_1.shape[0], interval=50)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# plot states and control sequence
def plot_drivers_comparison(
    x_hist_1,
    u_hist_1,
    x_hist_2,
    u_hist_2,
    dt,
    label1="Driver 1",
    label2="Driver 2",
    value1=None,
    value2=None,
    title="Driver Comparison",
):
    """
    Plots both drivers' trajectories on the same subplots for comparison.
    """
    # time
    t = jnp.arange(x_hist_1.shape[0]) * dt

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

    # Add value information to labels if provided
    label1_full = f"{label1}" + (f" (V={value1:.1f})" if value1 is not None else "")
    label2_full = f"{label2}" + (f" (V={value2:.1f})" if value2 is not None else "")

    # Position plot
    ax1.plot(t, x_hist_1[:, 0], label=label1_full, color="blue", linewidth=2)
    ax1.plot(t, x_hist_2[:, 0], label=label2_full, color="red", linewidth=2)
    ax1.set_ylabel("Position (m)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_title(title)

    # Velocity plot
    ax2.plot(t, x_hist_1[:, 1], label=label1_full, color="blue", linewidth=2)
    ax2.plot(t, x_hist_2[:, 1], label=label2_full, color="red", linewidth=2)
    ax2.set_ylabel("Velocity (m/s)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Control plot
    ax3.plot(t, u_hist_1, label=label1_full, color="blue", linewidth=2)
    ax3.plot(t, u_hist_2, label=label2_full, color="red", linewidth=2)
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Control (m/s²)")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# plot drivers comparison using history data, dt, values
plot_drivers_comparison(
    x_hist_1,
    u_hist_1,
    x_hist_2,
    u_hist_2,
    dt,
    label1="Aggressive Driver",
    label2="Gentle Driver",
    value1=V_test_1,
    value2=V_test_2,
    title="Driver Comparison: Aggressive vs Gentle (Starting from [4.0, 0.0])",
)

## (d) Race!
The drivers perform four different races, and three judges will give them a score based on their personal judging preferences.

The "Starting Positions" ($ x_0 $):

- $ [-15.0,\, 0.0] $ (Starting far away, at rest)

- $ [-5.0,\, 0.0] $ (Starting close, at rest)

- $ [0.0,\, 3.0] $ (At the finish line, but speeding!)

- $ [-8.0,\, 2.0] $ (A blend of the above)

In [ ]:
start_state_list = jnp.array(
    [
        [-15.0, 0.0],
        [-5.0, 0.0],
        [0.0, 3.0],
        [-8.0, 2.0],
    ]
)

### b-(i) Judge #1

Suppose that the first judge cares A LOT about reaching the finish line exactly, and also cares (a little less) about stopping at the finish line. But they don't care too much about the amount of control effort. Concretely, the first judge's preferences corresponds to the following parameters.

\begin{aligned}
Q=\text{diag}(50,\, 20),\quad R = 0.1
\end{aligned}

[TODO]: in peers/hw

Knowing this information about the first judge, before the race, can you guess which driver this judge would deem the winner? Briefly justify your guess. That is, predict who the winner would be according to the first judge without running the code below yet.

[student answers here]

Let's see what this judge decides on!

In [ ]:
# [TODO]: in peers/hw
Q_judge = jnp.diag(jnp.array([..., ...])) # change me
R_judge = ... # change me


print(f"{'Starting x0':<12} | {'Driver 1 (V)':<15} | {'Driver 2 (V)':<15}")
print("-" * 70)

# 3. Loop over each starting position and find the winner
for i, x0 in enumerate(start_state_list):
    # --- YOUR CODE HERE ---
    # Compute the value for Driver 1
    V_driver1, _ = ...

    # Compute the value for Driver 2
    V_driver2, _ = ...
    # ----------------------

    winner = "Driver 1" if V_driver1 < V_driver2 else "Driver 2"

    print(f"{str(x0):<12} | {V_driver1:<15.2f} | {V_driver2:<15.2f} | Winner: {winner}")

print("-" * 70)

### b-(ii) Judge #2

Now we have the second judge. They care a fair bit about finishing at the finish line, but don't care much about stopping there. But they do really care about control effort. Concretely, the second judge's preferences corresponds to the following parameters.

\begin{aligned}
Q=\text{diag}(5.0,\, 0.5),\quad R = 50
\end{aligned}


[TODO]: in peers/hw

Knowing this information about the first judge, before the race, can you guess which driver this judge would deem the winner? Briefly justify your guess. That is, predict who the winner would be according to the first judge without running the code below yet.

[student answers here]

Let's see what this judge decides on!

In [ ]:
# [TODO]: in peers/hw
Q_judge = jnp.diag(jnp.array([..., ...])) # change me
R_judge = ... # change me


print(f"{'Starting x0':<12} | {'Driver 1 (V)':<15} | {'Driver 2 (V)':<15}")
print("-" * 70)

# 3. Loop over each starting position and find the winner
for i, x0 in enumerate(start_state_list):
    # --- YOUR CODE HERE ---
    # Compute the value for Driver 1
    V_driver1, _ = ...

    # Compute the value for Driver 2
    V_driver2, _ = ...
    # ----------------------

    winner = "Driver 1" if V_driver1 < V_driver2 else "Driver 2"

    print(f"{str(x0):<12} | {V_driver1:<15.2f} | {V_driver2:<15.2f} | Winner: {winner}")

print("-" * 70)

### b-(iii) The third and final judge

This third judge is a bit more discerning, and care about all criteria more evenly.

\begin{aligned}
Q=\text{diag}(5.0,\, 2.0),\quad R = 0.5
\end{aligned}


[TODO]: in peers/hw

Knowing this information about the first judge, before the race, can you guess which driver this judge would deem the winner? Briefly justify your guess. That is, predict who the winner would be according to the first judge without running the code below yet.

[student answers here]

Let's see what this judge decides on!

In [ ]:
# [TODO]: in peers/hw
Q_judge = jnp.diag(jnp.array([..., ...])) # change me
R_judge = ... # change me


print(f"{'Starting x0':<12} | {'Driver 1 (V)':<15} | {'Driver 2 (V)':<15}")
print("-" * 70)

# 3. Loop over each starting position and find the winner
for i, x0 in enumerate(start_state_list):
    # --- YOUR CODE HERE ---
    # Compute the value for Driver 1
    V_driver1, _ = ...

    # Compute the value for Driver 2
    V_driver2, _ = ...
    # ----------------------

    winner = "Driver 1" if V_driver1 < V_driver2 else "Driver 2"

    print(f"{str(x0):<12} | {V_driver1:<15.2f} | {V_driver2:<15.2f} | Winner: {winner}")

print("-" * 70)

### (e) More powerful car

[TODO by student]

If the cars were more powerful and had a larger acceleration/braking limit $ u_{\max} $, say $u_{\max}=6$ would the outcomes be different? Briefly explain why or why not.

[student answer here]